<a href="https://colab.research.google.com/github/prattayabsmrstu/Wetland_Advance/blob/main/Copy_of_Wetlands_final_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee

try:
    project_id = 'wetland-514509'
    ee.Initialize(project=project_id)
    print("Google Earth Engine successfully initialized with project ID wetland-514509.")
except Exception as e:
    print("Authentication required. Please authenticate your account.")
    ee.Authenticate()
    ee.Initialize(project=project_id)
    print("Authentication and initialization completed successfully.")

# গোপালগঞ্জ জেলার বাউন্ডারি নির্ধারণ
gopalganj = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh")) \
    .filter(ee.Filter.eq("ADM2_NAME", "Gopalganj"))

# সেন্টিনেল ওয়ান সার ডেটাসেট ফিল্টার করা
s1_collection = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(gopalganj) \
    .filterDate("2025-01-01", "2025-12-31") \
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")) \
    .filter(ee.Filter.eq("instrumentMode", "IW"))

# ইমেজ কম্পোজিট তৈরি এবং ক্লিপ করা
s1_image = s1_collection.select("VH").mean().clip(gopalganj)

print("Gopalganj spatial boundary and Sentinel data extraction prepared successfully.")

Google Earth Engine successfully initialized with project ID wetland-514509.
Gopalganj spatial boundary and Sentinel data extraction prepared successfully.


/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


In [ ]:
import ee

# ১. আধুনিক GAUL লেভেল ২ ডেটাসেট ব্যবহার করে গোপালগঞ্জ জেলার বাউন্ডারি নির্ধারণ
gopalganj = ee.FeatureCollection("FAO/GAUL/2025/level2") \
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh")) \
    .filter(ee.Filter.eq("ADM2_NAME", "Gopalganj"))

# ২. সেন্টিনেল ওয়ান সার (Sentinel-1 SAR) ডেটাসেট ফিল্টার ও ওয়াটার মাস্ক এক্সট্রাকশন
s1_collection = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(gopalganj) \
    .filterDate("2025-01-01", "2025-12-31") \
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")) \
    .filter(ee.Filter.eq("instrumentMode", "IW"))

# ভিএইচ ব্যান্ডের মিন কম্পোজিট তৈরি এবং ক্লিপ করা
s1_image = s1_collection.select("VH").mean().clip(gopalganj)

# ওটসু থ্রেশহোল্ডিং বা নির্দিষ্ট থ্রেশহোল্ড (যেমন: -15 dB) ব্যবহার করে ওয়াটার বডি আইসোলেশন
# সাধারণত SAR ব্যাকস্ক্যাটারের ক্ষেত্রে কম মান (Dark pixels) পানি নির্দেশ করে
water_mask = s1_image.lt(-15).rename('water_extent')

# ৩. কোপারনিকাস DEM (30m) থেকে এলিভেশন এবং স্লোপ এক্সট্রাকশন
dem = ee.Image("COPERNICUS/DEM/GLO30").select('DEM').clip(gopalganj)
slope = ee.Terrain.slope(dem).rename('slope')

# ৪. সম্মিলিত স্ট্যাক তৈরি (PINN মডেলের ইনপুট স্প্যাশিয়াল ডেটার জন্য)
model_spatial_features = dem.addBands(slope).addBands(water_mask).addBands(s1_image.rename('sar_vh'))

print("Step 2 Completed: Topographic DEM, Slope, and SAR-derived Water Mask successfully generated for Gopalganj.")

Step 2 Completed: Topographic DEM, Slope, and SAR-derived Water Mask successfully generated for Gopalganj.


/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/DEM/GLO30! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/DEM/GLO30_2024_1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30

  warnings.warn(warning, category=DeprecationWarning)


In [4]:
import ee
import geemap

# ১. প্রজেক্ট ইনিশিয়ালাইজেশন
project_id = 'wetland-514509'
try:
    ee.Initialize(project=project_id)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=project_id)

# ২. গোপালগঞ্জ জেলার প্রশাসনিক বাউন্ডারি (FAO GAUL 2015 ডেটাসেট)
gopalganj = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh")) \
    .filter(ee.Filter.eq("ADM2_NAME", "Gopalganj"))

# ৩. সেন্টিনেল-১ সার ডেটাসেট (২০১৫ থেকে ২০২৫) ফিল্টার ও ওয়াটার মাস্ক এক্সট্রাকশন
s1_collection = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(gopalganj) \
    .filterDate("2015-01-01", "2025-12-31") \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))

count = s1_collection.size().getInfo()
print(f"Total Sentinel-1 images found (2015-2025): {count}")

if count > 0:
    s1_image = s1_collection.select("VH").mean().clip(gopalganj)
    water_mask = s1_image.lt(-15).rename('water_extent')
else:
    water_mask = ee.Image.constant(0).rename('water_extent').clip(gopalganj)

# কোপারনিকাস ডিইএম কালেকশন থেকে মোজাইক তৈরি ও ক্লিপিং
dem = ee.ImageCollection("COPERNICUS/DEM/GLO30").select('DEM').mosaic().clip(gopalganj)
slope = ee.Terrain.slope(dem).rename('slope')

# ৪. ইন্টারেক্টিভ ম্যাপ তৈরি (কোনো বেসম্যাপ ব্যবহার করা হয়নি যাতে 403 ত্রুটি না আসে)
Map = geemap.Map()
Map.centerObject(gopalganj, 10)  # গোপালগঞ্জ জেলার ওপর স্বয়ংক্রিয় জুম

# ভিজ্যুয়ালাইজেশন প্যারামিটার
dem_vis = {'min': 0, 'max': 20, 'palette': ['blue', 'green', 'yellow', 'brown']}
water_vis = {'min': 0, 'max': 1, 'palette': ['white', 'navy']}

# গোপালগঞ্জ অ্যাডমিন বাউন্ডারি স্টাইল (লাল রঙের স্পষ্ট বর্ডার, ভেতরের অংশ স্বচ্ছ)
boundary_styled = gopalganj.style(color='red', fillColor='00000000', width=2)

# ম্যাপে শুধুমাত্র আমাদের রিকোয়ারড লেয়ারগুলো যুক্ত করা
Map.addLayer(dem, dem_vis, 'Copernicus DEM (2015-2025)')
Map.addLayer(water_mask, water_vis, 'SAR Long-term Water Extent')
Map.addLayer(boundary_styled, {}, 'Gopalganj Admin Boundary')

Map

Total Sentinel-1 images found (2015-2025): 825


Map(center=[23.104257654279518, 89.89886218568263], controls=(WidgetControl(options=['position', 'transparent_…